# 03 — Building a Fintech Agent with Claude Tool Use

This notebook demonstrates how to build a multi-tool financial agent using
the Anthropic SDK's tool use feature. The agent can look up stock data,
calculate ratios, and generate chart data.

**Prerequisites:**
```bash
pip install anthropic
export ANTHROPIC_API_KEY="sk-ant-..."
```

In [ ]:
import sys, json
sys.path.insert(0, '../agents')
from tool_use_financial import TOOLS, TOOL_DISPATCH, run_agent

## 1. Inspect the tools

The agent has three tools. Claude sees the tool definitions and decides when to call them.

In [ ]:
for tool in TOOLS:
    print(f"Tool: {tool['name']}")
    print(f"  {tool['description'][:100]}...")
    print()

## 2. Single-turn examples

Each call lets Claude decide which tools to use (if any) and returns a final answer.

In [ ]:
# Simple lookup — triggers get_stock_price
response = run_agent("What's the current price of AAPL?")
print(response)

In [ ]:
# Ratio calculation — triggers calculate_ratios
response = run_agent(
    "Calculate the debt-to-equity ratio for a company with $1.2B total debt "
    "and $3.5B total equity."
)
print(response)

In [ ]:
# Multi-tool — may trigger both lookup and chart generation
response = run_agent(
    "Compare the P/E ratios of AAPL, GOOGL, and MSFT. "
    "Which looks most attractively valued? Generate a bar chart of the comparison."
)
print(response)

## 3. How the agent loop works

The pattern is:
1. Send the user message + tool definitions to Claude
2. If Claude returns `tool_use` blocks, execute them locally
3. Send the tool results back to Claude
4. Repeat until Claude returns a final text response

This loop is implemented in `agents/tool_use_financial.py` — see the `run_agent()` function.

## 4. Extending with your own tools

To add a new tool:
1. Add its definition to `TOOLS` (name, description, input_schema)
2. Implement the function
3. Register it in `TOOL_DISPATCH`

For example, you could add a `fetch_sec_filing` tool that retrieves real EDGAR data.

In [ ]:
# Example: adding a custom tool definition
custom_tool = {
    'name': 'fetch_sec_filing',
    'description': 'Retrieve a SEC filing (10-K, 10-Q, 8-K) for a given company and year.',
    'input_schema': {
        'type': 'object',
        'properties': {
            'ticker': {'type': 'string', 'description': 'Company ticker'},
            'filing_type': {'type': 'string', 'enum': ['10-K', '10-Q', '8-K']},
            'year': {'type': 'integer'},
        },
        'required': ['ticker', 'filing_type', 'year'],
    },
}

print(json.dumps(custom_tool, indent=2))